In [26]:
# Loading PDF
import pymupdf
from typing import List, Dict, Any

def extract_pdf_to_structured_list(pdf_path: str) -> List[Dict[str, Any]]:
    structured_data = []
    
    with pymupdf.open(pdf_path) as doc:
        for page_num, page in enumerate(doc):
            # Extract basic text
            page_text = page.get_text()
            
            # Construct a rich metadata dictionary for each page
            page_dict = {
                "page_number": page_num + 1,
                "total_pages": doc.page_count,
                "text": page_text,
                "char_count": len(page_text),
                "word_count": len(page_text.split()),
                "dimensions": {
                    "width": page.rect.width,
                    "height": page.rect.height
                },
                # Optional: Extract links or images present on this specific page
                "links": page.get_links()
            }
            
            structured_data.append(page_dict)
            
    return structured_data

# Execute the loader
pdf_data = extract_pdf_to_structured_list("data/Advanced_Business_Seller_Guide_May09.pdf")

# Example: Inspecting the first page's structured structure
print(pdf_data[0])



#chunking PDF

from typing import List, Dict, Any


def chunk_documents(
    documents: List[Dict[str, Any]],
    chunk_size: int = 1000,
    overlap: int = 200
) -> List[Dict[str, Any]]:

    # Validate input
    if not isinstance(documents, list):
        raise TypeError("documents must be a list of dictionaries")

    if chunk_size <= 0:
        raise ValueError("chunk_size must be > 0")

    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and < chunk_size")

    chunks = []
    stride = chunk_size - overlap

    for doc in documents:
        if not isinstance(doc, dict):
            raise TypeError("Each document must be a dictionary")

        required_keys = ["content", "title", "description", "file_name"]
        if not all(key in doc for key in required_keys):
            raise ValueError(f"Document missing one of required keys: {required_keys}")

        content = doc["content"]
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string")

        for i in range(0, len(content), stride):
            chunk_text = content[i : i + chunk_size]

            chunks.append(
                {
                    "content": chunk_text,
                    "title": doc["title"],
                    "description": doc["description"],
                    "file_name": doc["file_name"],
                }
            )

    return chunks


# Demo usage
if __name__ == "__main__":
    docs = pdf_data

    out = chunk_documents(docs, chunk_size=1000, overlap=200)
    print(f"Generated {len(out)} chunks")
    for c in out:
        print(len(c["content"]))

{'page_number': 1, 'total_pages': 24, 'text': '1\nAdvanced Business \nSeller Guide\n', 'char_count': 34, 'word_count': 5, 'dimensions': {'width': 612.0, 'height': 792.0}, 'links': []}


ValueError: Document missing one of required keys: ['content', 'title', 'description', 'file_name']